# 09 · Hosted agents – when configuration is not enough

A **prompt agent** is configuration. A **hosted agent** is *your code* – any framework, any library, custom logic – packaged and run by Foundry with an endpoint, an Entra identity, scaling, tracing and session storage.

| | Prompt agent | Hosted agent |
|---|---|---|
| You write | definition (model, instructions, tools) | code (Agent Framework, LangGraph, your own) |
| Runs | inside Foundry's agent runtime | in a managed container/sandbox |
| Tools | built-in, MCP, A2A, functions you loop yourself | anything your code can call |
| Deploy | `create_version` | `azd deploy` (code ZIP or container) |
| Protocols | Responses | Responses, Invocations, A2A, ... |

⚠️ Deploying needs the **Foundry Project Manager** role on the project (formerly *Azure AI Project Manager*). `python scripts/grant_hosted_role.py` assigns it to you. Hosted agents are a newer feature: check the [docs](https://learn.microsoft.com/azure/foundry/agents/quickstarts/quickstart-hosted-agent) if a command differs. Deep dive: [docs/hosted-agents.md](../docs/hosted-agents.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_replies import show_reply, escalation_checks, skill_checks, status_checks

HOSTED_AGENT_NAME = "jarvis-hosted"    # services in azure.yaml: skills pasted into the instructions
SKILLS_AGENT_NAME = "jarvis-skills"    # skills loaded on demand (section 4)

print((ROOT / "src" / "jarvis-hosted" / "main.py").read_text())

**What the code does**

1. `FoundryChatClient` talks to the model through the project (identity: yours locally, the **agent identity** when hosted).
2. `Agent` adds instructions, the local tool `workshop_status` and the skills from `skills/` (direct injection, chapter 07): every call carries every skill. Section 4 deploys a second agent that loads skills on demand instead.
3. `ResponsesHostServer` exposes the agent over the **Responses protocol**: Foundry stores conversations and handles streaming.

## 1 · Install the azd extension and run locally
```bash
python scripts/grant_hosted_role.py          # once: Foundry Project Manager role for you
azd extension install microsoft.foundry      # adds: azd ai agent ...
python scripts/sync_skills.py                # copy skills/ into src/jarvis-hosted/skills
azd ai agent run                             # starts main.py locally and opens the agent inspector
azd ai agent invoke --local "Status of the arc reactor?"
```

## 2 · Deploy
`azd deploy` zips `src/jarvis-hosted`, Foundry builds it remotely (`codeConfiguration` in [azure.yaml](../azure.yaml)) and creates version 1. No Docker needed in code mode.

In [ ]:
sh("azd deploy jarvis-hosted --no-prompt")

## 3 · Call it like any agent
Hosted agents answer the same Responses API as prompt agents: an agent-bound client, a conversation, `responses.create`. Two questions, each aimed at one part of `main.py`:

| Question | Text | Should use |
|---|---|---|
| `STATUS_QUESTION` | What is the status of the arc reactor? | the tool `workshop_status` |
| `INCIDENT_QUESTION` | A suit just started smoking in the hangar. What do I do? | the `stark-escalation` skill in the instructions |

The next cell opens `hosted_conversation`; cases A and B both run in it. Running a case again adds a turn to that conversation, so a repeated answer may refer to the earlier one. Section 4 reuses both questions.

In [ ]:
hosted_client = openai_client(agent_name=HOSTED_AGENT_NAME)
hosted_conversation = hosted_client.conversations.create()
STATUS_QUESTION = "What is the status of the arc reactor?"
INCIDENT_QUESTION = "A suit just started smoking in the hangar. What do I do?"
print("conversation:", hosted_conversation.id)

### Case A · A status question

**Goal:** the local tool works inside the container. Only `workshop_status` knows the workshop's state.

**Expected:** the answer reports the arc reactor as online, 3.0 GW, last service 2026-09-30: the `WORKSHOP` values in `main.py`. The check looks for `3.0 GW` in the final message. **Tool calls** lists the `workshop_status` call: the hosted agent passes its tool calls through in the response (seen on a live project).

`show_reply()` ([stark_replies.py](stark_replies.py)) shows the prompt, the tool calls, the agent's whole answer and the check table.

In [ ]:
status_reply = hosted_client.responses.create(conversation=hosted_conversation.id, input=STATUS_QUESTION)
show_reply(STATUS_QUESTION, status_reply, status_checks(status_reply), to=HOSTED_AGENT_NAME)

### Case B · An incident

**Goal:** prove the skill reached the container. `jarvis-hosted` pastes every `SKILL.md` into its instructions at startup.

**Expected:** the final message starts with `Severity: critical`, tells you to evacuate and notify Pepper Potts. The check table:

| | |
|---|---|
| severity first (final message) | yes (expected yes) ✅ |
| severity (final message) | `critical` (expected `critical`) ✅ |
| names a floor (final message) | yes or no |

The floor has no expected value: `azd deploy` copied the repo's [SKILL.md](../skills/stark-escalation/SKILL.md), which has no version 2 addendum. Chapter 07 added that only to the skill in Foundry.

In [ ]:
incident_reply = hosted_client.responses.create(conversation=hosted_conversation.id, input=INCIDENT_QUESTION)
show_reply(INCIDENT_QUESTION, incident_reply,
           escalation_checks(incident_reply, expected={"severity first": True, "severity": "critical"}), to=HOSTED_AGENT_NAME)

## 4 · Skills on demand: `jarvis-skills`
[src/jarvis-skills/main.py](../src/jarvis-skills/main.py) is the same J.A.R.V.I.S. with one change: skills go through Agent Framework's **`SkillsProvider`** instead of into the instructions.

| | `jarvis-hosted` | `jarvis-skills` |
|---|---|---|
| Skills reach the model | all of them, pasted into the instructions | name and description only; the body when the model calls `load_skill` |
| Workshop status | the tool `workshop_status` | the code-defined skill `workshop-status` (a resource and a script) |
| Cost per call | grows with every skill you add | grows only with the skills the model loads |
| Proof the skill was used | the answer | the answer, plus `load_skill` calls in the response output and the traces |

All three skill tools (`load_skill`, `read_skill_resource`, `run_skill_script`) need approval by default. The agent turns approval off because the skills are this repo's own. Keep it on for skills from elsewhere.

Deploy it, then ask the same two questions (`STATUS_QUESTION`, `INCIDENT_QUESTION` from section 3) without a conversation. The check table now also lists which skills the model loaded.

> 💡 To draft a skill without deploying, run `python samples/skills_provider.py "your question"`: the same agent as a plain script, printing which skill tools the model called.


In [ ]:
sh("azd deploy jarvis-skills --no-prompt")

> ✅ **Tested on a live project.** Deployed with `azd deploy`, `jarvis-skills` loaded the right skill in cases C and D below. Before that, the code was tested locally: `ResponsesHostServer` returned the `function_call` items for `load_skill`, `read_skill_resource` and `run_skill_script` with a scripted model.

### Case C · The status question, skills on demand

**Goal:** the model finds the code-defined `workshop-status` skill from its name and description alone, and loads it.

**Expected:** a `load_skill` call for `workshop-status`, then probably `read_skill_resource` (`systems`) and `run_skill_script` (`status`), because the skill's instructions say so. The answer reports 3.0 GW, as in case A.

| | |
|---|---|
| skills loaded | `workshop-status` (expected `workshop-status`) ✅ |
| reports `3.0 GW` (final message) | yes (expected yes) ✅ |

In [ ]:
skills_client = openai_client(agent_name=SKILLS_AGENT_NAME)
skills_status_reply = skills_client.responses.create(input=STATUS_QUESTION)
show_reply(STATUS_QUESTION, skills_status_reply,
           {**skill_checks(skills_status_reply, expected_skill="workshop-status"), **status_checks(skills_status_reply)}, to=SKILLS_AGENT_NAME)

### Case D · The incident, skills on demand

**Goal:** the same skill as case B, but the model has to ask for it. Only its name and description were in the prompt.

**Expected:** a `load_skill` call for `stark-escalation`, then the same markers as case B: the final message starts with `Severity: critical`.

| | |
|---|---|
| skills loaded | `stark-escalation` (expected `stark-escalation`) ✅ |
| severity first (final message) | yes (expected yes) ✅ |
| severity (final message) | `critical` (expected `critical`) ✅ |
| names a floor (final message) | yes or no |

In [ ]:
skills_incident_reply = skills_client.responses.create(input=INCIDENT_QUESTION)
show_reply(INCIDENT_QUESTION, skills_incident_reply,
           {**skill_checks(skills_incident_reply, expected_skill="stark-escalation"),
            **escalation_checks(skills_incident_reply, expected={"severity first": True, "severity": "critical"})}, to=SKILLS_AGENT_NAME)

## 5 · Observe it
```bash
azd ai agent monitor --follow     # container logs
azd ai agent show                 # versions, endpoint, protocols
```
Hosted agents get `APPLICATIONINSIGHTS_CONNECTION_STRING` from the platform, so their traces land in the same Application Insights as chapter 03 – find the conversation id in **Traces**.

## 👀 In the portal
🧪 **Build → Agents → jarvis-hosted**: a *hosted* agent. Its page lists the versions that `azd deploy` created, the playground and **Traces** with the conversation id.

**Build → Agents → jarvis-skills**: the same, and its **Traces** show the `load_skill` tool call when the model loads a skill (seen on a live project).

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
hosted_client.conversations.delete(hosted_conversation.id)
# delete_agent(HOSTED_AGENT_NAME)   # uncomment to remove the hosted agent; `azd down --purge` removes everything
# delete_agent(SKILLS_AGENT_NAME)   # uncomment to remove the skills-on-demand agent

## 🎯 Recap
Prompt agents for speed and simplicity; hosted agents when you need code. Skills ride along either way: pasted into the instructions (`jarvis-hosted`) or loaded on demand by `SkillsProvider` (`jarvis-skills`). Both are first-class Foundry agents: same conversations, tracing, and evaluations (next act).

> 💡 Next: Act III – **F.R.I.D.A.Y.** proves it works. [10 · Datasets and evaluations](10_datasets_and_evals.ipynb).